In [0]:
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType, DoubleType, BooleanType, DateType
from pyspark.sql import functions as F

In [0]:
storage_account = "tokyoolympicdatazul"
container = "tokyoolympicdata"

configs = {
    f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net": "OAuth",

    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net":
        "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",

    f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net":
        "ClientID",

    f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net":
        "SecretKey",

    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net":
        "https://login.microsoftonline.com/TenantID/oauth2/token"
}

# Apply the OAuth configuration to Spark
for key, value in configs.items():
    spark.conf.set(key, value)

# ADLS Gen2 path
path = f"abfss://tokyoolympicdata@tokyoolympicdatazul.dfs.core.windows.net/"

# Read data
df = spark.read.csv(
    path,
    header=True,
    inferSchema=True
)


In [0]:
display(dbutils.fs.ls(
    "abfss://tokyoolympicdata@tokyoolympicdatazul.dfs.core.windows.net/"
))

path,name,size,modificationTime
abfss://tokyoolympicdata@tokyoolympicdatazul.dfs.core.windows.net/raw_data/,raw_data/,0,1789159706000
abfss://tokyoolympicdata@tokyoolympicdatazul.dfs.core.windows.net/transformed_data/,transformed_data/,0,1789159717000


In [0]:
athletes = spark.read.csv(
    "abfss://tokyoolympicdata@tokyoolympicdatazul.dfs.core.windows.net/raw_data/athletes.csv",
    header=True,
    inferSchema=True
)
coaches = spark.read.csv(
    "abfss://tokyoolympicdata@tokyoolympicdatazul.dfs.core.windows.net/raw_data/coaches.csv",
    header=True,
    inferSchema=True
)
entriesgender = spark.read.csv(
    "abfss://tokyoolympicdata@tokyoolympicdatazul.dfs.core.windows.net/raw_data/entriesgender.csv",
    header=True,
    inferSchema=True
)
medals = spark.read.csv(
    "abfss://tokyoolympicdata@tokyoolympicdatazul.dfs.core.windows.net/raw_data/medals.csv",
    header=True,
    inferSchema=True
)
teams = spark.read.csv(
    "abfss://tokyoolympicdata@tokyoolympicdatazul.dfs.core.windows.net/raw_data/teams.csv",
    header=True,
    inferSchema=True
)

In [0]:
athletes.show(5)
athletes.printSchema()

+-----------------+-------+-------------------+
|       PersonName|Country|         Discipline|
+-----------------+-------+-------------------+
|  AALERUD Katrine| Norway|       Cycling Road|
|      ABAD Nestor|  Spain|Artistic Gymnastics|
|ABAGNALE Giovanni|  Italy|             Rowing|
|   ABALDE Alberto|  Spain|         Basketball|
|    ABALDE Tamara|  Spain|         Basketball|
+-----------------+-------+-------------------+
only showing top 5 rows
root
 |-- PersonName: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Discipline: string (nullable = true)



In [0]:
coaches.show(5)
coaches.printSchema()

+---------------+-------------+----------+-----+
|           Name|      Country|Discipline|Event|
+---------------+-------------+----------+-----+
|ABDELMAGID Wael|        Egypt|  Football| NULL|
|      ABE Junya|        Japan|Volleyball| NULL|
|  ABE Katsuhiko|        Japan|Basketball| NULL|
|   ADAMA Cherif|C�te d'Ivoire|  Football| NULL|
|     AGEBA Yuya|        Japan|Volleyball| NULL|
+---------------+-------------+----------+-----+
only showing top 5 rows
root
 |-- Name: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Discipline: string (nullable = true)
 |-- Event: string (nullable = true)



In [0]:
entriesgender.show(5)
entriesgender.printSchema()

+-------------------+------+----+-----+
|         Discipline|Female|Male|Total|
+-------------------+------+----+-----+
|     3x3 Basketball|    32|  32|   64|
|            Archery|    64|  64|  128|
|Artistic Gymnastics|    98|  98|  196|
|  Artistic Swimming|   105|   0|  105|
|          Athletics|   969|1072| 2041|
+-------------------+------+----+-----+
only showing top 5 rows
root
 |-- Discipline: string (nullable = true)
 |-- Female: integer (nullable = true)
 |-- Male: integer (nullable = true)
 |-- Total: integer (nullable = true)



In [0]:
medals.show(5)
medals.printSchema()

+----+--------------------+----+------+------+-----+-------------+
|Rank|         TeamCountry|Gold|Silver|Bronze|Total|Rank by Total|
+----+--------------------+----+------+------+-----+-------------+
|   1|United States of ...|  39|    41|    33|  113|            1|
|   2|People's Republic...|  38|    32|    18|   88|            2|
|   3|               Japan|  27|    14|    17|   58|            5|
|   4|       Great Britain|  22|    21|    22|   65|            4|
|   5|                 ROC|  20|    28|    23|   71|            3|
+----+--------------------+----+------+------+-----+-------------+
only showing top 5 rows
root
 |-- Rank: integer (nullable = true)
 |-- TeamCountry: string (nullable = true)
 |-- Gold: integer (nullable = true)
 |-- Silver: integer (nullable = true)
 |-- Bronze: integer (nullable = true)
 |-- Total: integer (nullable = true)
 |-- Rank by Total: integer (nullable = true)



In [0]:
teams.show(5)
teams.printSchema()

+--------+--------------+--------------------+-----+
|TeamName|    Discipline|             Country|Event|
+--------+--------------+--------------------+-----+
| Belgium|3x3 Basketball|             Belgium|  Men|
|   China|3x3 Basketball|People's Republic...|  Men|
|   China|3x3 Basketball|People's Republic...|Women|
|  France|3x3 Basketball|              France|Women|
|   Italy|3x3 Basketball|               Italy|Women|
+--------+--------------+--------------------+-----+
only showing top 5 rows
root
 |-- TeamName: string (nullable = true)
 |-- Discipline: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Event: string (nullable = true)



In [0]:
top_gold_medal_countries = medals.orderBy("Gold", ascending=False).select("TeamCountry","Gold").show()

+--------------------+----+
|         TeamCountry|Gold|
+--------------------+----+
|United States of ...|  39|
|People's Republic...|  38|
|               Japan|  27|
|       Great Britain|  22|
|                 ROC|  20|
|           Australia|  17|
|         Netherlands|  10|
|              France|  10|
|             Germany|  10|
|               Italy|  10|
|                Cuba|   7|
|         New Zealand|   7|
|              Brazil|   7|
|              Canada|   7|
|             Hungary|   6|
|   Republic of Korea|   6|
|               Kenya|   4|
|              Poland|   4|
|      Czech Republic|   4|
|              Norway|   4|
+--------------------+----+
only showing top 20 rows


In [0]:
# Calculate the average number of entries by gender for each discipline
average_entries_by_gender = entriesgender.withColumn(
    'Avg_Female', entriesgender['Female'] / entriesgender['Total']
).withColumn(
    'Avg_Male', entriesgender['Male'] / entriesgender['Total']
)
average_entries_by_gender.show()

+--------------------+------+----+-----+-------------------+-------------------+
|          Discipline|Female|Male|Total|         Avg_Female|           Avg_Male|
+--------------------+------+----+-----+-------------------+-------------------+
|      3x3 Basketball|    32|  32|   64|                0.5|                0.5|
|             Archery|    64|  64|  128|                0.5|                0.5|
| Artistic Gymnastics|    98|  98|  196|                0.5|                0.5|
|   Artistic Swimming|   105|   0|  105|                1.0|                0.0|
|           Athletics|   969|1072| 2041| 0.4747672709456149| 0.5252327290543851|
|           Badminton|    86|  87|  173|0.49710982658959535| 0.5028901734104047|
|   Baseball/Softball|    90| 144|  234|0.38461538461538464| 0.6153846153846154|
|          Basketball|   144| 144|  288|                0.5|                0.5|
|    Beach Volleyball|    48|  48|   96|                0.5|                0.5|
|              Boxing|   102

In [0]:
base_path = "abfss://tokyoolympicdata@tokyoolympicdatazul.dfs.core.windows.net/"

dataframes = {
    "athletes": athletes,
    "medals": medals,
    "entriesgender": entriesgender,
    "teams": teams,
    "coaches": coaches
}

for name, df in dataframes.items():

    # Temporary Spark output folder
    temp_path = f"{base_path}temp/{name}"

    # Final output file
    final_path = f"{base_path}transformed_data/{name}.csv"

    # Write as a single CSV
    df.repartition(1).write \
        .mode("overwrite") \
        .option("header", "true") \
        .csv(temp_path)

    # Find the generated CSV file
    csv_file = [
        f.path
        for f in dbutils.fs.ls(temp_path)
        if f.path.endswith(".csv")
    ][0]

    # Delete existing final file if present
    dbutils.fs.rm(final_path, True)

    # Move/rename to desired filename
    dbutils.fs.mv(csv_file, final_path)

    # Remove temporary folder and Spark metadata
    dbutils.fs.rm(temp_path, True)

    print(f"{name}.csv completed")

athletes.csv completed
medals.csv completed
entriesgender.csv completed
teams.csv completed
coaches.csv completed
